# Amazon Bedrock Batch and Async Jobs

This notebook demonstrates Amazon Bedrock batch and asynchronous job patterns, including `StartAsyncInvoke` and `CreateModelInvocationJob`. It also includes best practices for job monitoring and S3 output handling.

In [ ]:
import os
import json
import time
import boto3

region = os.getenv('AWS_REGION', 'us-east-1')
bedrock = boto3.client('bedrock', region_name=region)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=region)

print('Clients initialized for', region)

## CreateModelInvocationJob Example

This example shows how to submit a batch job that reads input data from S3 and writes output back to S3.

In [ ]:
job_name = 'bedrock-batch-sample'

response = bedrock.create_model_invocation_job(
    roleArn='arn:aws:iam::<account>:role/<bedrock-invoke-role>',
    modelId='anthropic.claude-3-haiku-20240307-v1:0',
    jobName=job_name,
    inputDataConfig={
        's3InputDataConfig': {
            's3Uri': 's3://<bucket_name>/<input_file>.jsonl',
        },
    },
    outputDataConfig={
        's3OutputDataConfig': {
            's3Uri': 's3://<bucket_name>/<output-prefix>/',
        },
    },
)
print('Created job ARN:', response.get('jobArn'))
print('Job status:', response.get('status'))

## Poll Job Status

When a batch job is created, poll the status until it is complete or failed.

In [ ]:
job_arn = response.get('jobArn')

for attempt in range(20):
    status_response = bedrock.get_model_invocation_job(jobArn=job_arn)
    status = status_response.get('status')
    print(f'Attempt {attempt + 1}:', status)
    if status in {'COMPLETED', 'FAILED', 'CANCELLED'}:
        break
    time.sleep(15)

## StartAsyncInvoke Example

This pattern is useful for video and other long-running generation jobs.

In [ ]:
async_response = bedrock_runtime.start_async_invoke(
    modelId='amazon.nova-reel-v1:0',
    modelInput={
        'taskType': 'TEXT_VIDEO',
        'textToVideoParams': {'text': 'A calm sunrise over a mountain lake.'},
        'videoGenerationConfig': {
            'fps': 24,
            'durationSeconds': 5,
            'dimension': '1280x720',
        },
    },
    outputDataConfig={
        's3OutputDataConfig': {'s3Uri': 's3://<bucket_name>/<prefix>/'},
    },
)
print('Async invoke ARN:', async_response.get('invocationArn'))

## Best Practices

- Use an IAM role with least privilege for Bedrock batch jobs.
- Validate S3 URIs and permissions before submitting a job.
- Keep retries and polling intervals reasonable to avoid throttling.
- Record job ARNs and outputs for auditability.
- Redact any sensitive content from input datasets before sending to Bedrock.